# Notebook 01 — Data Collection

Assembles raw data and writes three CSVs that Notebook 02 consumes:
- `data/raw/grants_raw.csv` — individual grant records (funder → recipient, amount, purpose)
- `data/raw/funders_raw.csv` — funder (foundation) summaries
- `data/raw/recipients_raw.csv` — unique grant recipients (name, city, state)

**Sources**
1. **IRS 990-PF index CSVs** (`apps.irs.gov`) — fast population counts of e-filed 990-PFs
2. **IRS 990-PF XML bulk ZIP** (`apps.irs.gov`) — the actual grant detail (one ~400 MB chunk)
3. **ProPublica Nonprofit Explorer API** — optional funder enrichment

> **Notes on the IRS data**
> - The IRS deprecated its AWS S3 e-file bucket in Dec 2021; data now comes as bulk ZIP
>   archives that mix 990 / 990-EZ / 990-PF, so we filter to 990-PF after download.
> - 990-PF grant records list recipient **name + address but no EIN** (the form doesn't
>   require it), so recipients are keyed by name + state. EIN-based enrichment of recipients
>   (NTEE, demographics) is left to the Candid integration.

All raw outputs are gitignored. **Run top to bottom**, then go to Notebook 02.

In [ ]:
import sys
sys.path.insert(0, '..')

import time
import logging
from pathlib import Path

import pandas as pd
from tqdm.notebook import tqdm

from src.api_client import (
    fetch_990_index,
    sample_990pf_from_zip,
    parse_990pf_grants,
    parse_990pf_funder_summary,
    propublica_organization,
)

logging.basicConfig(level=logging.INFO)
RAW = Path('../data/raw')
RAW.mkdir(parents=True, exist_ok=True)
print('Setup complete.')

## 1  IRS 990-PF Index (fast population counts)

The index CSVs are small and list every e-filed 990-PF for a year. We use them to show the
size of the filer population by year. (Grant *detail* comes from the XML in section 2.)

In [ ]:
YEARS = [2018, 2019, 2020, 2021]   # index years to summarize

counts = {}
for year in YEARS:
    counts[year] = len(fetch_990_index(year))
    print(f"{year}: {counts[year]:,} 990-PF filings")

pd.Series(counts, name='filings').to_csv(RAW / '990pf_index_counts.csv')
print('\nSaved filer population counts.')

## 2  Download & parse 990-PF XML grant data

Downloads one bulk ZIP chunk (~400 MB, **cached** so it only happens once), filters to 990-PF
returns, and parses their grant schedules. `SAMPLE_SIZE` caps how many 990-PF filings we take.

Produces `grants_raw.csv` and `funders_raw.csv`.

In [ ]:
SAMPLE_SIZE = 2000   # number of 990-PF filings to take from the chunk
ZIP_YEAR = 2020      # processing-year archive
CHUNK = '1'          # 2020 chunks are '1'..'8'

filings = sample_990pf_from_zip(ZIP_YEAR, chunk=CHUNK, max_filings=SAMPLE_SIZE, save=True)
print(f"Collected {len(filings)} 990-PF filings")

In [ ]:
grants_records = []
funder_records = []

for oid, xml_text in tqdm(filings.items(), desc='Parsing 990-PF XML'):
    try:
        gs = parse_990pf_grants(xml_text)
        for g in gs:
            g['object_id'] = oid
        grants_records.extend(gs)
        funder_records.append(parse_990pf_funder_summary(xml_text))
    except Exception as e:
        print(f"parse error {oid}: {e}")

grants_raw = pd.DataFrame(grants_records)
funders_raw = pd.DataFrame(funder_records)
grants_raw.to_csv(RAW / 'grants_raw.csv', index=False)
funders_raw.to_csv(RAW / 'funders_raw.csv', index=False)

print(f"grants_raw:  {grants_raw.shape[0]:,} grant records")
print(f"funders_raw: {funders_raw.shape[0]:,} foundations")
grants_raw.head()

## 3  Unique recipients

990-PF filings give recipient name + city + state (no EIN), so we build a recipient table
keyed by normalized name + state. Produces `recipients_raw.csv`.

In [ ]:
if grants_raw.empty:
    raise RuntimeError('grants_raw is empty — re-run section 2 first.')

recip = (
    grants_raw[['recipient_name', 'recipient_city', 'recipient_state']]
    .rename(columns={'recipient_name': 'name', 'recipient_city': 'city', 'recipient_state': 'state'})
    .dropna(subset=['name'])
    .drop_duplicates(['name', 'state'])
    .reset_index(drop=True)
)
# Columns kept for schema compatibility with Notebook 02 (filled via Candid later)
recip['ein'] = None
recip['ntee_code'] = None
recip['total_revenue'] = None
recip.to_csv(RAW / 'recipients_raw.csv', index=False)

print(f"recipients_raw: {len(recip):,} unique recipients")
recip.head()

## 4  ProPublica funder enrichment (optional)

Demonstrates the ProPublica API client by pulling profiles for a few well-known foundations.
Not required by downstream notebooks.

In [ ]:
SEED_EINS = {
    '131684331': 'Ford Foundation',
    '237093598': 'John D. & Catherine T. MacArthur Foundation',
    '381359264': 'W.K. Kellogg Foundation',
    '131659629': 'Rockefeller Foundation',
    '521951681': 'Annie E. Casey Foundation',
}
rows = []
for ein, label in SEED_EINS.items():
    try:
        org = propublica_organization(ein).get('organization', {})
        rows.append({'ein': ein, 'name': org.get('name'), 'state': org.get('state'),
                     'ntee_code': org.get('ntee_code'), 'revenue': org.get('revenue_amount')})
        time.sleep(0.3)
    except Exception as e:
        rows.append({'ein': ein, 'name': label, 'error': str(e)})

pd.DataFrame(rows).to_csv(RAW / 'foundations_propublica.csv', index=False)
pd.DataFrame(rows)

## 5  Candid APIs (stub)

Activates once `CANDID_API_KEY` is set in a `.env` file at the project root.
Candid would supply recipient EINs, NTEE codes, and leadership demographics.

In [ ]:
# from dotenv import load_dotenv
# load_dotenv('../.env')
# from src.api_client import candid_demographics
print('Candid API stub — set CANDID_API_KEY in .env to activate')

## Summary

Written to `data/raw/`: `990pf_index_counts.csv`, `grants_raw.csv`, `funders_raw.csv`,
`recipients_raw.csv`, `foundations_propublica.csv`.

Proceed to **Notebook 02** for cleaning and SQLite loading.